# NB05 — Train PPO (Apple Full-Body) — SOTA Edition

**Env:** `UnitreeG1PlaceAppleInBowlFullBody-v1`  
**Control:** `pd_joint_delta_pos` (37 DOF, fixed root)  
**Reward:** `normalized_dense` (ground-truth normalization from env)

SOTA upgrades:
- Explicit `reward_mode="normalized_dense"`
- Correct env registration via `apple_fullbody_env.py`
- `SiLU` + split actor/critic nets
- PPO-safe `VecNormalize` (obs-only; reward norm OFF)
- Optional action safety: jerk limiter + EMA + action-scale warmup
- Robust eval utilities that work with **Gym envs** and **SB3 VecEnv/VecNormalize**
- Best checkpoint by success_rate + GIF recorder + failure taxonomy

Last updated: 2026-03-03 05:53


## 0) Imports & paths

In [1]:

import os, sys, json, time, random, inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import torch

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed

try:
    import imageio.v2 as imageio
except Exception:
    imageio = None

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

# Robust project-root detection (works in VS Code, RunPod, Colab)
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path(".").resolve()
    PROJECT_ROOT = _cwd
    for _c in [_cwd, _cwd.parent, _cwd.parent.parent]:
        if (_c / "src").is_dir():
            PROJECT_ROOT = _c
            break

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB05_PPO_SOTA"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)


/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/sapien/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


PROJECT_ROOT: /root/robotic-sim-dishwash
ARTIFACTS_DIR: /root/robotic-sim-dishwash/artifacts/NB05_PPO_SOTA


## 1) Register custom env

In [2]:

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_env_registered = False
try:
    import src.envs as _src_envs  # registers all custom envs via @register_env decorators
    _env_registered = True
except Exception:
    pass

if not _env_registered:
    import importlib.util as _ilu
    _p = PROJECT_ROOT / "src" / "envs" / "apple_fullbody_env.py"
    if not _p.exists():
        _p = PROJECT_ROOT / "apple_fullbody_env.py"
    assert _p.exists(), f"❌ apple_fullbody_env.py not found under {PROJECT_ROOT}"
    _spec = _ilu.spec_from_file_location("apple_fullbody_env", str(_p))
    _mod = _ilu.module_from_spec(_spec)
    _spec.loader.exec_module(_mod)
    _env_registered = True

print("✅ Custom env registered")
print("   PROJECT_ROOT:", PROJECT_ROOT)
print("   sys.path[0]:", sys.path[0])


✅ Custom env registered
   PROJECT_ROOT: /root/robotic-sim-dishwash
   sys.path[0]: /root/robotic-sim-dishwash


## 2) Configuration

In [3]:

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"

CFG = {
    "seed": 42,

    # Env
    "env_id":        "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "obs_mode":      "state",
    "control_mode":  "pd_joint_delta_pos",
    "reward_mode":   "normalized_dense",

    # Parallelism — ManiSkill GPU-batched (num_envs inside gym.make, NOT SubprocVecEnv)
    # SAPIEN holds a GPU/IPC context that cannot cross process boundaries:
    # SubprocVecEnv always fails with "Broken pipe" / "Connection reset".
    # Correct pattern: gym.make(..., num_envs=N) → SAPIEN runs N envs in
    # parallel GPU kernels → CPUGymWrapper → DummyVecEnv([single_env]).
    "num_sapien_envs": 32,      # SAPIEN-internal batch size (GPU parallel)
    "n_vec_envs":      1,       # SB3 VecEnv slots (always 1 for this pattern)

    # Timesteps (best quality / compute)
    "total_env_steps": 12_000_000,

    # PPO — rollout = num_sapien_envs * n_steps = 32*256 = 8192
    "learning_rate": 1e-4,
    "lr_end": 2e-5,
    "n_steps": 256,
    "batch_size": 2048,
    "n_epochs": 4,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_range": 0.10,
    "ent_coef": 0.001,
    "vf_coef": 0.5,
    "max_grad_norm": 0.3,
    "target_kl": 0.03,

    # Policy net
    "activation": "SiLU",
    "net_arch_pi": [512, 256, 128],
    "net_arch_vf": [512, 256, 128],

    "use_sde": False,
    "sde_sample_freq": 4,

    # VecNormalize
    "use_vecnormalize": True,
    "norm_obs": True,
    "norm_reward": False,
    "clip_obs": 10.0,

    # Action safety
    "use_action_filter": True,
    "ema_alpha": 0.12,
    "jerk_clip": 0.06,

    # Curriculum Phase A
    "phaseA_steps":        2_000_000,
    "phaseA_posture_k":    1.2,
    "phaseA_posture_clip": 0.04,

    # Curriculum Phase B
    "phaseB_scale_hold":   0.08,
    "phaseB_posture_k":    0.8,
    "phaseB_posture_clip": 0.02,

    # Checkpoint & eval
    "checkpoint_freq": 500_000,
    "eval_freq":       250_000,
    "eval_episodes":   20,

    # GIF
    "record_gif":      True,
    "gif_fps":         20,
    "gif_max_frames":  300,
}

SEED = CFG["seed"]
set_random_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

(ARTIFACTS_DIR / "nb05_config.json").write_text(json.dumps(CFG, indent=2))
print("✅ Saved config:", ARTIFACTS_DIR / "nb05_config.json")
print(f"DEVICE: {DEVICE}  IS_GPU: {IS_GPU}")
print(f"num_sapien_envs={CFG['num_sapien_envs']}  n_steps={CFG['n_steps']}  "
      f"rollout_size={CFG['num_sapien_envs']*CFG['n_steps']:,}  batch_size={CFG['batch_size']}")
print(f"total_env_steps: {CFG['total_env_steps']:,}")
print(f"Phase A: {CFG['phaseA_steps']:,} steps  → Phase B: hold_scale={CFG['phaseB_scale_hold']}, k={CFG['phaseB_posture_k']}")
print()
print("NOTE: SubprocVecEnv is NOT used (SAPIEN GPU context breaks across processes).")
print("      Parallelism = gym.make(num_envs=32) → SAPIEN GPU batch → DummyVecEnv([1])")


✅ Saved config: /root/robotic-sim-dishwash/artifacts/NB05_PPO_SOTA/nb05_config.json
DEVICE: cuda  IS_GPU: True
num_sapien_envs=32  n_steps=256  rollout_size=8,192  batch_size=2048
total_env_steps: 12,000,000
Phase A: 2,000,000 steps  → Phase B: hold_scale=0.08, k=0.8

NOTE: SubprocVecEnv is NOT used (SAPIEN GPU context breaks across processes).
      Parallelism = gym.make(num_envs=32) → SAPIEN GPU batch → DummyVecEnv([1])


## 3) Action safety wrappers

In [4]:

from stable_baselines3.common.vec_env.base_vec_env import VecEnv as _VecEnvBase

# ── Gym wrappers (for eval_env / probe only) ──────────────────────────────────

class ActionFilterWrapper(gym.Wrapper):
    """EMA smoothing + per-joint jerk limit (single-env, for eval/probe)."""
    def __init__(self, env, ema_alpha=0.2, jerk_clip=0.08):
        super().__init__(env)
        self.ema_alpha = float(ema_alpha)
        self.jerk_clip = float(jerk_clip)
        self._prev = None
        self._ema  = None

    def reset(self, **kwargs):
        out = self.env.reset(**kwargs)
        self._prev = np.zeros(self.action_space.shape, dtype=np.float32)
        self._ema  = None
        return out

    def step(self, action):
        a = np.asarray(action, dtype=np.float32)
        if self._ema is None:
            self._ema = a.copy()
        else:
            self._ema = (1.0 - self.ema_alpha) * self._ema + self.ema_alpha * a
        d   = np.clip(self._ema - self._prev, -self.jerk_clip, self.jerk_clip)
        out = self._prev + d
        self._prev = out.copy()
        return self.env.step(out.astype(np.float32))

    def apply_phase_b(self, hold_scale, posture_k, posture_clip):
        e = self.env
        while e is not None:
            if isinstance(e, ActionMaskWrapper):
                e._hold_scale = float(hold_scale)
            if isinstance(e, PostureHoldAlignedWrapper):
                e.k    = float(posture_k)
                e.clip = float(posture_clip)
            e = getattr(e, "env", None)
        return True


class ActionMaskWrapper(gym.Wrapper):
    def __init__(self, env, free_indices):
        super().__init__(env)
        self._free       = np.asarray(sorted(set(free_indices)), dtype=int)
        self._hold_scale = 0.0

    def step(self, action):
        src = np.asarray(action, dtype=np.float32)
        a   = src.copy()
        hold_mask = np.ones(self.action_space.shape[0], dtype=bool)
        hold_mask[self._free] = False
        if self._hold_scale == 0.0:
            a[hold_mask] = 0.0
        else:
            a[hold_mask] *= self._hold_scale
        return self.env.step(a)


class PostureHoldAlignedWrapper(gym.Wrapper):
    def __init__(self, env, hold_indices, k=1.2, clip=0.04):
        super().__init__(env)
        self.hold_indices = np.asarray(hold_indices, dtype=int)
        self.k    = float(k)
        self.clip = float(clip)
        self._joints  = None
        self.qpos_ref = None

    def _cache_joints(self):
        robot = self.env.unwrapped.agent.robot
        for attr in ["active_joints", "get_active_joints"]:
            if hasattr(robot, attr):
                val = getattr(robot, attr)
                self._joints = list(val() if callable(val) else val)
                return
        self._joints = None

    def _get_qpos_vec(self):
        if self._joints is None:
            return self._fallback_qpos()
        qs = []
        for j in self._joints:
            try:
                fn = getattr(j, "get_qpos", None)
                v  = fn() if callable(fn) else getattr(j, "qpos", None)
                if v is None:
                    qs.append(0.0); continue
                if hasattr(v, "detach"):
                    v = v.detach().cpu().numpy()
                v = np.asarray(v, dtype=np.float32).reshape(-1)
                qs.append(float(v[0]) if len(v) else 0.0)
            except Exception:
                qs.append(0.0)
        act_dim = self.action_space.shape[0]
        arr = np.asarray(qs, dtype=np.float32)
        return arr[:act_dim] if arr.shape[0] >= act_dim else np.pad(arr, (0, act_dim - arr.shape[0]))

    def _fallback_qpos(self):
        robot = self.env.unwrapped.agent.robot
        for attr in ["get_qpos", "qpos"]:
            if hasattr(robot, attr):
                qpos = getattr(robot, attr)
                qpos = qpos() if callable(qpos) else qpos
                if hasattr(qpos, "detach"):
                    qpos = qpos.detach().cpu().numpy()
                qpos = np.asarray(qpos, dtype=np.float32).reshape(-1)
                act_dim = self.action_space.shape[0]
                return qpos[:act_dim] if qpos.shape[0] >= act_dim else np.pad(qpos, (0, act_dim - qpos.shape[0]))
        return None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._cache_joints()
        qpos = self._get_qpos_vec()
        self.qpos_ref = qpos.copy() if qpos is not None else None
        return obs, info

    def step(self, action):
        a = np.asarray(action, dtype=np.float32)
        if self.qpos_ref is not None and self.k > 0.0:
            qpos = self._get_qpos_vec()
            if qpos is not None:
                n  = min(len(a), len(qpos), len(self.qpos_ref))
                hi = self.hold_indices[self.hold_indices < n]
                err = qpos[:n][hi] - self.qpos_ref[:n][hi]
                hold = np.zeros_like(a)
                hold[hi] = np.clip(-self.k * err, -self.clip, self.clip)
                a = a + hold
        return self.env.step(a.astype(np.float32))


# ── ManiSkillVecEnv: SB3 VecEnv for GPU-batched ManiSkill ────────────────────
#
# SAPIEN holds a GPU/IPC context that cannot cross process boundaries.
# SubprocVecEnv always fails (Broken pipe / Connection reset) with SAPIEN.
#
# Correct pattern:
#   gym.make(num_envs=N, render_backend="gpu")
#   → N SAPIEN envs in parallel GPU kernels, single process, zero IPC overhead.
#   → Wrap with ManiSkillVecEnv (SB3 VecEnv subclass).
#
# Built-in batched: ActionFilter + ActionMask + PostureHold
# Curriculum via apply_phase_b() through training_env.env_method().

class ManiSkillVecEnv(_VecEnvBase):
    """
    SB3 VecEnv adapter for ManiSkill GPU-batched environments.

    gym.make(num_envs=N, render_backend="gpu") runs N SAPIEN envs in
    parallel on GPU in one process — no subprocess, no IPC, no broken pipe.

    Built-in batched pipeline (per step):
      policy_action (N,37)
        → ActionFilter (batched EMA + jerk, state reset on episode done)
        → ActionMask   (hold_scale=0.0 Phase A → soft Phase B)
        → PostureHold  (vectorised PD correction toward reset qpos)
        → env.step()

    Phase A → Phase B via apply_phase_b() called through env_method().
    """

    def __init__(
        self,
        ms_env,
        free_indices,
        hold_indices,
        phaseA_posture_k: float    = 1.2,
        phaseA_posture_clip: float = 0.04,
        use_filter: bool           = True,
        ema_alpha: float           = 0.12,
        jerk_clip: float           = 0.06,
    ):
        self._ms_env = ms_env
        n      = ms_env.unwrapped.num_envs
        obs_sp = ms_env.unwrapped.single_observation_space
        act_sp = ms_env.unwrapped.single_action_space
        super().__init__(n, obs_sp, act_sp)

        self._free = np.asarray(free_indices, dtype=int)
        self._hold = np.asarray(hold_indices, dtype=int)

        # Phase A defaults
        self._hold_scale   = 0.0
        self._posture_k    = float(phaseA_posture_k)
        self._posture_clip = float(phaseA_posture_clip)
        self._qpos_ref     = None     # (N, ACT_DIM) — set at reset

        # ActionFilter state
        self._use_filter = use_filter
        self.ema_alpha   = float(ema_alpha)
        self.jerk_clip   = float(jerk_clip)
        self._ema  = None             # (N, ACT_DIM)
        self._prev = None             # (N, ACT_DIM)

        self._actions_buf = None

    # ── helpers ───────────────────────────────────────────────────────────────
    @staticmethod
    def _np(x):
        if torch.is_tensor(x):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    def _get_qpos(self) -> np.ndarray:
        """(N, ACT_DIM) qpos as numpy."""
        qpos = self._ms_env.unwrapped.agent.robot.get_qpos()
        return self._np(qpos).reshape(self.num_envs, -1)

    # ── action pipeline ────────────────────────────────────────────────────────
    def _filter(self, a: np.ndarray) -> np.ndarray:
        if not self._use_filter:
            return a
        if self._ema is None:
            self._ema  = a.copy()
            self._prev = a.copy()
            return a
        self._ema = (1.0 - self.ema_alpha) * self._ema + self.ema_alpha * a
        d   = np.clip(self._ema - self._prev, -self.jerk_clip, self.jerk_clip)
        out = self._prev + d
        self._prev = out.copy()
        return out

    def _mask(self, a: np.ndarray) -> np.ndarray:
        out = a.copy()
        hold_mask = np.ones(a.shape[1], dtype=bool)
        hold_mask[self._free] = False
        if self._hold_scale == 0.0:
            out[:, hold_mask] = 0.0
        else:
            out[:, hold_mask] *= self._hold_scale
        return out

    def _posture_hold(self, a: np.ndarray) -> np.ndarray:
        if self._qpos_ref is None or self._posture_k <= 0.0:
            return a
        qpos = self._get_qpos()
        n  = min(a.shape[1], qpos.shape[1], self._qpos_ref.shape[1])
        hi = self._hold[self._hold < n]
        err  = qpos[:, hi] - self._qpos_ref[:, hi]
        corr = np.clip(-self._posture_k * err, -self._posture_clip, self._posture_clip)
        out = a.copy()
        out[:, hi] += corr
        return out

    # ── SB3 VecEnv API ────────────────────────────────────────────────────────
    def reset(self) -> np.ndarray:
        obs, _ = self._ms_env.reset()
        obs_np = self._np(obs).reshape(self.num_envs, -1)
        self._qpos_ref = self._get_qpos()
        self._ema = self._prev = None
        return obs_np

    def step_async(self, actions: np.ndarray) -> None:
        a = np.asarray(actions, dtype=np.float32).reshape(self.num_envs, -1)
        a = self._filter(a)
        a = self._mask(a)
        a = self._posture_hold(a)
        self._actions_buf = a

    def step_wait(self):
        obs, rew, term, trunc, info = self._ms_env.step(self._actions_buf)
        obs_np  = self._np(obs).reshape(self.num_envs, -1)
        rew_np  = self._np(rew).reshape(-1).astype(np.float32)
        term_np = self._np(term).reshape(-1).astype(bool)
        trunc_np= self._np(trunc).reshape(-1).astype(bool)
        done_np = term_np | trunc_np

        # ManiSkill GPU env auto-resets done envs before returning.
        # Update qpos_ref and EMA for done envs from the new episode.
        if done_np.any() and self._qpos_ref is not None:
            new_qpos = self._get_qpos()
            self._qpos_ref[done_np] = new_qpos[done_np]
            if self._ema is not None:
                self._ema[done_np]  = 0.0
                self._prev[done_np] = 0.0

        # Build per-env info dicts (SB3 expects list[dict])
        infos = []
        for i in range(self.num_envs):
            d = {}
            for k, v in info.items():
                try:
                    arr = self._np(v).reshape(-1)
                    d[k] = arr[i] if len(arr) > i else arr[0]
                except Exception:
                    d[k] = v
            if done_np[i]:
                d["terminal_observation"] = obs_np[i]
            infos.append(d)

        return obs_np, rew_np, done_np, infos

    def close(self) -> None:
        self._ms_env.close()

    def render(self, mode="human"):
        ret = self._ms_env.render()
        return self._np(ret) if ret is not None else None

    # ── Curriculum ────────────────────────────────────────────────────────────
    def apply_phase_b(self, hold_scale: float, posture_k: float, posture_clip: float):
        """CurriculumCallback calls: training_env.env_method('apply_phase_b', ...)."""
        self._hold_scale   = float(hold_scale)
        self._posture_k    = float(posture_k)
        self._posture_clip = float(posture_clip)
        return True

    # ── SB3 VecEnv required stubs ─────────────────────────────────────────────
    def env_is_wrapped(self, wrapper_class, indices=None):
        return [False] * self.num_envs

    def env_method(self, method_name, *method_args, indices=None, **method_kwargs):
        method = getattr(self, method_name, None)
        result = method(*method_args, **method_kwargs) if method is not None else None
        n = self.num_envs if indices is None else len(indices)
        return [result] * n

    def get_attr(self, attr_name, indices=None):
        val = getattr(self, attr_name, None)
        n = self.num_envs if indices is None else len(indices)
        return [val] * n

    def set_attr(self, attr_name, value, indices=None):
        setattr(self, attr_name, value)

    def seed(self, seed=None):
        return [seed] * self.num_envs


print("✅ Gym wrappers defined (ActionFilter, ActionMask, PostureHoldAligned) — used by eval_env")
print("✅ ManiSkillVecEnv defined — GPU-batched train_env, no subprocess, built-in mask+posture+filter")
print("   SubprocVecEnv is NOT used (SAPIEN GPU context cannot cross process boundaries)")


✅ Gym wrappers defined (ActionFilter, ActionMask, PostureHoldAligned) — used by eval_env
✅ ManiSkillVecEnv defined — GPU-batched train_env, no subprocess, built-in mask+posture+filter
   SubprocVecEnv is NOT used (SAPIEN GPU context cannot cross process boundaries)


## 4) Env factory

In [5]:

# ── Step 1: Load action_groups ─────────────────────────────────────────────
_ag_path = PROJECT_ROOT / "artifacts" / "NB02" / "action_groups.json"
assert _ag_path.exists(), f"❌ action_groups.json not found at {_ag_path}"
ACTION_GROUPS = json.loads(_ag_path.read_text())
print("✅ Loaded action_groups.json:", {g: len(v) for g, v in ACTION_GROUPS.items()})

# ── Step 2: Create GPU batch env FIRST ────────────────────────────────────
# CRITICAL: SAPIEN PhysX can only be initialised once per process.
# A CPU probe env (num_envs=1, render_backend="none") starts CPU PhysX,
# which then blocks enable_gpu() for the GPU batch env.
# Solution: create the GPU batch env BEFORE any other env.
N_TRAIN_ENVS = CFG["num_sapien_envs"]

_backend_used = "gpu"
try:
    _ms_train = gym.make(
        CFG["env_id"], num_envs=N_TRAIN_ENVS,
        obs_mode=CFG["obs_mode"], reward_mode=CFG["reward_mode"],
        control_mode=CFG["control_mode"],
        render_mode="rgb_array", render_backend="gpu",
    )
    print(f"✅ ManiSkill GPU batch  num_envs={N_TRAIN_ENVS}")
except Exception as _e:
    print(f"⚠️  GPU backend unavailable ({type(_e).__name__}), using CPU batch")
    _backend_used = "cpu"
    _ms_train = gym.make(
        CFG["env_id"], num_envs=N_TRAIN_ENVS,
        obs_mode=CFG["obs_mode"], reward_mode=CFG["reward_mode"],
        control_mode=CFG["control_mode"],
        render_mode=None, render_backend="cpu",
    )
    print(f"   CPU batch  num_envs={N_TRAIN_ENVS}")

# Reset once so robot state is available for joint-name extraction
_ms_train.reset()

# ── Step 3: Derive joint indices from the GPU env (no separate probe env) ─
# Joint names are identical for every parallel env; read from env 0 of the batch.
def _get_joint_names_from_batched(ms_env):
    robot   = ms_env.unwrapped.agent.robot
    act_dim = ms_env.unwrapped.single_action_space.shape[0]
    for attr in ["active_joints", "get_active_joints", "joints", "get_joints"]:
        if hasattr(robot, attr):
            joints = getattr(robot, attr)
            if callable(joints):
                joints = joints()
            names = []
            for j in joints:
                nm = getattr(j, "name", None) or (j.get_name() if hasattr(j, "get_name") else None)
                names.append(str(nm))
            return names[:act_dim]
    raise RuntimeError("Cannot locate robot joints API on batched env")

def build_indices_from_names(names, action_groups):
    n2i = {n: i for i, n in enumerate(names)}
    g2idx, missing = {}, {}
    for g, jlist in action_groups.items():
        idxs = [n2i[jn] for jn in jlist if jn in n2i]
        miss = [jn for jn in jlist if jn not in n2i]
        g2idx[g]   = sorted(set(idxs))
        if miss:
            missing[g] = miss
    return g2idx, missing

JOINT_NAMES        = _get_joint_names_from_batched(_ms_train)
GROUP2IDX, MISSING = build_indices_from_names(JOINT_NAMES, ACTION_GROUPS)

if MISSING:
    print("⚠️  Missing joints:", MISSING)

FREE    = sorted(set(GROUP2IDX.get("right_arm", []) + GROUP2IDX.get("right_hand", [])))
HOLD    = sorted([i for i in range(len(JOINT_NAMES)) if i not in set(FREE)])
ACT_DIM = len(JOINT_NAMES)

print(f"action_dim={ACT_DIM} | free={len(FREE)} | hold={len(HOLD)}")
print(f"torso in HOLD? {all(i in HOLD for i in GROUP2IDX.get('torso', []))}")

# ── Step 4: Wrap in ManiSkillVecEnv ───────────────────────────────────────
train_env = ManiSkillVecEnv(
    _ms_train, FREE, HOLD,
    phaseA_posture_k    = CFG["phaseA_posture_k"],
    phaseA_posture_clip = CFG["phaseA_posture_clip"],
    use_filter          = CFG["use_action_filter"],
    ema_alpha           = CFG["ema_alpha"],
    jerk_clip           = CFG["jerk_clip"],
)
train_env.reset()

# ── Step 5: eval_env ── CPU single env (GPU PhysX already enabled above) ─
# num_envs=1 CPU env runs with CPU simulation — does NOT call enable_gpu().
# This is created AFTER the GPU env so there's no PhysX conflict.
RENDER_AVAILABLE = True

def _build_single_env(seed: int, for_eval: bool = False):
    """Single CPU env with gym wrapper chain (for eval/probe only)."""
    global RENDER_AVAILABLE
    def _init():
        global RENDER_AVAILABLE
        try:
            e = gym.make(
                CFG["env_id"], num_envs=1,
                obs_mode=CFG["obs_mode"], reward_mode=CFG["reward_mode"],
                control_mode=CFG["control_mode"],
                render_mode="rgb_array", render_backend="cpu",
            )
        except RuntimeError:
            RENDER_AVAILABLE = False
            e = gym.make(
                CFG["env_id"], num_envs=1,
                obs_mode=CFG["obs_mode"], reward_mode=CFG["reward_mode"],
                control_mode=CFG["control_mode"],
                render_mode=None, render_backend="none",
            )
        e = CPUGymWrapper(e)
        e = PostureHoldAlignedWrapper(e, HOLD,
                k=CFG["phaseA_posture_k"], clip=CFG["phaseA_posture_clip"])
        e = ActionMaskWrapper(e, FREE)
        if CFG["use_action_filter"]:
            e = ActionFilterWrapper(e, CFG["ema_alpha"], CFG["jerk_clip"])
        e.reset(seed=seed)
        return e
    return _init

eval_env = DummyVecEnv([_build_single_env(SEED + 9999, for_eval=True)])

# ── Step 6: VecNormalize ──────────────────────────────────────────────────
if CFG["use_vecnormalize"]:
    train_env = VecNormalize(
        train_env, norm_obs=CFG["norm_obs"], norm_reward=CFG["norm_reward"],
        clip_obs=CFG["clip_obs"], training=True,
    )
    eval_env = VecNormalize(
        eval_env, norm_obs=CFG["norm_obs"], norm_reward=False,
        clip_obs=CFG["clip_obs"], training=False,
    )
    eval_env.obs_rms = train_env.obs_rms

print()
print(f"train_env : ManiSkillVecEnv({_backend_used} x{N_TRAIN_ENVS})  → {type(train_env).__name__}")
print(f"eval_env  : DummyVecEnv(CPU x1)  → {type(eval_env).__name__}")
print(f"render_available={RENDER_AVAILABLE}")
print(f"rollout_size = {N_TRAIN_ENVS} × {CFG['n_steps']} = {N_TRAIN_ENVS * CFG['n_steps']:,} transitions/update")
print()
print(f"Phase A: hold_scale=0.0, k={CFG['phaseA_posture_k']}  ({CFG['phaseA_steps']:,} steps)")
print(f"Phase B: hold_scale={CFG['phaseB_scale_hold']}, k={CFG['phaseB_posture_k']}  (via env_method)")


✅ Loaded action_groups.json: {'left_leg': 6, 'right_leg': 6, 'torso': 1, 'left_arm': 5, 'right_arm': 5, 'left_hand': 7, 'right_hand': 7}


⚠️  GPU backend unavailable (RuntimeError), using CPU batch


[2026-03-03 12:07:52.562] [SAPIEN] [warning] loading multiple convex collision meshes from STL file is unsupported and can result in invalid collision meshes. Do you mean to load a single convex mesh instead?
[2026-03-03 12:07:52.617] [SAPIEN] [warning] loading multiple convex collision meshes from STL file is unsupported and can result in invalid collision meshes. Do you mean to load a single convex mesh instead?
[2026-03-03 12:07:52.629] [SAPIEN] [warning] loading multiple convex collision meshes from STL file is unsupported and can result in invalid collision meshes. Do you mean to load a single convex mesh instead?
[2026-03-03 12:07:52.648] [SAPIEN] [warning] loading multiple convex collision meshes from STL file is unsupported and can result in invalid collision meshes. Do you mean to load a single convex mesh instead?
[2026-03-03 12:07:52.691] [SAPIEN] [warning] loading multiple convex collision meshes from STL file is unsupported and can result in invalid collision meshes. Do yo

   CPU batch  num_envs=32
action_dim=37 | free=12 | hold=25
torso in HOLD? True

train_env : ManiSkillVecEnv(cpu x32)  → VecNormalize
eval_env  : DummyVecEnv(CPU x1)  → VecNormalize
render_available=True
rollout_size = 32 × 256 = 8,192 transitions/update

Phase A: hold_scale=0.0, k=1.2  (2,000,000 steps)
Phase B: hold_scale=0.08, k=0.8  (via env_method)


[2026-03-03 12:07:56.532] [SAPIEN] [warning] A PhysX CPU system is being created while PhysX GPU is enabled. You can safely ignore this message if it is intended. To use GPU PhysX, create a sapien.physx.PhysxGpuSystem explicitly and pass it to sapien.Scene constructor.


In [ ]:

# ── Chain inspector ────────────────────────────────────────────────────────

def _unwrap_one(e):
    if hasattr(e, "venv"):   return e.venv
    if hasattr(e, "envs"):   return e.envs[0]
    if hasattr(e, "env"):    return e.env
    return None

def _walk_chain(root, label=""):
    print(f"=== {label} wrapper chain ===")
    e, depth = root, 0
    while e is not None:
        print(f"  {'  '*depth}{depth}: {type(e).__name__}")
        depth += 1
        e = _unwrap_one(e)
    print()

def _find_by_name(root, cls_name):
    e = root
    while e is not None:
        if type(e).__name__ == cls_name:
            return e
        e = _unwrap_one(e)
    return None

def _depth_of(root, cls_name):
    e, depth = root, 0
    while e is not None:
        if type(e).__name__ == cls_name:
            return depth
        depth += 1
        e = _unwrap_one(e)
    return -1

# ── Verify train_env (ManiSkillVecEnv) ────────────────────────────────────
# ManiSkillVecEnv wraps a single GPU-batched env — no per-env gym chain.
# Inspect state attributes directly instead of traversing a wrapper chain.
print("=== train_env (ManiSkillVecEnv) ===")
# Unwrap VecNormalize to get ManiSkillVecEnv
_ms_vec = train_env.venv if hasattr(train_env, "venv") else train_env
print(f"  outer    : {type(train_env).__name__}")
print(f"  inner    : {type(_ms_vec).__name__}")
if isinstance(_ms_vec, ManiSkillVecEnv):
    print(f"  num_envs      : {_ms_vec.num_envs}")
    print(f"  _hold_scale   : {_ms_vec._hold_scale}  (0.0 = Phase A hard-zero ✅)")
    print(f"  _posture_k    : {_ms_vec._posture_k}")
    print(f"  _posture_clip : {_ms_vec._posture_clip}")
    print(f"  _use_filter   : {_ms_vec._use_filter}  ema_alpha={_ms_vec.ema_alpha}")
    print(f"  FREE joints   : {len(_ms_vec._free)}  HOLD joints: {len(_ms_vec._hold)}")
    _qref = _ms_vec._qpos_ref
    if _qref is not None:
        print(f"  _qpos_ref     : shape={_qref.shape}  norm={np.linalg.norm(_qref):.4f} ✅")
    else:
        print(f"  _qpos_ref     : None (call train_env.reset() first)")
    print(f"  apply_phase_b : {'✅' if hasattr(_ms_vec, 'apply_phase_b') else '❌'}")
else:
    print(f"  ⚠️  Expected ManiSkillVecEnv, got {type(_ms_vec).__name__}")
print()

# ── Verify eval_env (DummyVecEnv gym chain) ───────────────────────────────
print("=== eval_env (DummyVecEnv + gym wrappers) ===")
_walk_chain(eval_env, "eval_env")

_probe_eval = eval_env.venv if hasattr(eval_env, "venv") else eval_env
ph   = _find_by_name(_probe_eval, "PostureHoldAlignedWrapper")
mask = _find_by_name(_probe_eval, "ActionMaskWrapper")
af   = _find_by_name(_probe_eval, "ActionFilterWrapper")
d_filt = _depth_of(_probe_eval, "ActionFilterWrapper")
d_mask = _depth_of(_probe_eval, "ActionMaskWrapper")
d_hold = _depth_of(_probe_eval, "PostureHoldAlignedWrapper")

if ph and mask:
    if d_hold > d_mask:
        print("  ✅ eval_env order: ActionFilter → ActionMask → PostureHold(innermost)")
    else:
        print("  ❌ eval_env order WRONG")
    print(f"  PostureHold k={ph.k}  qpos_ref={'SET' if ph.qpos_ref is not None else 'None'}")
    print(f"  ActionMask  hold_scale={mask._hold_scale}")
print()

# ── Distance check on eval_env ────────────────────────────────────────────
print("--- distance check ---")
eval_env.reset()
_d_t2a = dist_tcp_apple(eval_env)
_d_a2b = dist_apple_bowl(eval_env)
print(f"   dist_tcp_apple : {_d_t2a}")
print(f"   dist_apple_bowl: {_d_a2b}")
if _d_t2a is not None:
    print("   ✅ KPI read works — no NaN")

print()
print(f"✅ Verification complete")
print(f"   train_env: ManiSkillVecEnv x{N_TRAIN_ENVS} (GPU batch, no subprocess)")
print(f"   eval_env : DummyVecEnv x1 (CPU, direct env.unwrapped access)")
print(f"   Phase B triggers at {CFG['phaseA_steps']:,} steps via env_method('apply_phase_b')")


--- building probe env for chain inspection ---


=== probe (= 1 train env) wrapper chain ===
  0: DummyVecEnv
    1: ActionFilterWrapper
      2: ActionMaskWrapper
        3: PostureHoldAlignedWrapper
          4: CPUGymWrapper
            5: TimeLimitWrapper
              6: OrderEnforcing
                7: UnitreeG1PlaceAppleInBowlFullBodyEnv

✅ PostureHoldAlignedWrapper  k=1.2  clip=0.04
   qpos_ref SET  shape=(37,)  norm=3.0558
   hold_indices[:5]=[0 1 2 3 4]  total=25
✅ ActionMaskWrapper  free=12 joints  hold_scale=0.0 (Phase A=0.0) ✅
✅ ActionFilterWrapper.apply_phase_b() exists (SubprocVecEnv curriculum ready)

  Depths (higher = closer to env):
    ActionFilter    depth=1
    ActionMask      depth=2
    PostureHold     depth=3
  ✅ ORDER CORRECT: ActionFilter → ActionMask → PostureHold(innermost)
     PostureHold corrections will NOT be zeroed by ActionMask ✅

✅ Chain verified. train_env has 32 envs. Phase A active (hold_scale=0.0, k=1.2)
   Phase B triggers at 2,000,000 steps via env_method('apply_phase_b', ...)
   (Distance 

/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.venv to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.venv` for environment variables or `env.get_wrapper_attr('venv')` that will search the reminding wrappers.
  logger.warn(
/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.envs to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.envs` for environment variables or `env.get_wrapper_attr('envs')` that will search the reminding wrappers.
  logger.warn(


## 5) Robust eval utilities (Gym env + SB3 VecEnv)

In [ ]:

def _to_np(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        x = np.array(x)
    if isinstance(x, np.ndarray) and x.ndim == 2 and x.shape[0] == 1:
        x = x[0]
    return x

def _scalar(x) -> float:
    """Safely convert any scalar/array/tensor to a Python float."""
    return float(np.atleast_1d(np.asarray(_to_np(x), dtype=float)).ravel()[0])


# ── Direct env.unwrapped KPI helpers (guaranteed non-NaN) ─────────────────
# Per INSTRUCTION.md: do NOT rely on obs/info for distances — read directly.

def _get_raw_env(vec_env):
    """
    Traverse VecNormalize → DummyVecEnv → Gym wrapper chain → base env.
    Returns the innermost Gym env (call .unwrapped for the actual SAPIEN env).
    """
    e = vec_env
    while e is not None:
        if hasattr(e, "venv"):   # VecNormalize, VecFrameStack, etc.
            e = e.venv
        elif hasattr(e, "envs"): # DummyVecEnv, SubprocVecEnv
            e = e.envs[0]
        elif hasattr(e, "env"):  # gym.Wrapper chain
            e = e.env
        else:
            break
    return e

def get_tcp_pos(vec_env):
    """Read TCP (end-effector) position from env.unwrapped._tcp_link.pose.p."""
    try:
        raw = _get_raw_env(vec_env).unwrapped
        p = raw._tcp_link.pose.p
        return np.asarray(_to_np(p), dtype=np.float32).reshape(-1)[:3]
    except Exception:
        return None

def get_apple_pos(vec_env):
    """Read apple position from env.unwrapped.apple.pose.p."""
    try:
        raw = _get_raw_env(vec_env).unwrapped
        p = raw.apple.pose.p
        return np.asarray(_to_np(p), dtype=np.float32).reshape(-1)[:3]
    except Exception:
        return None

def get_bowl_pos(vec_env):
    """Read bowl position from env.unwrapped.bowl.pose.p."""
    try:
        raw = _get_raw_env(vec_env).unwrapped
        p = raw.bowl.pose.p
        return np.asarray(_to_np(p), dtype=np.float32).reshape(-1)[:3]
    except Exception:
        return None

def dist_tcp_apple(vec_env):
    """Euclidean distance TCP → apple. Returns float or None."""
    tp = get_tcp_pos(vec_env)
    ap = get_apple_pos(vec_env)
    if tp is None or ap is None:
        return None
    return float(np.linalg.norm(tp - ap))

def dist_apple_bowl(vec_env):
    """Euclidean distance apple → bowl centre. Returns float or None."""
    ap = get_apple_pos(vec_env)
    bp = get_bowl_pos(vec_env)
    if ap is None or bp is None:
        return None
    return float(np.linalg.norm(ap - bp))


# ── Env utility functions ──────────────────────────────────────────────────

def get_extra(obs, info):
    if isinstance(info, dict) and any(k in info for k in ["tcp_to_apple","apple_to_bowl","is_grasped"]):
        out = {}
        for k in ["tcp_to_apple","apple_to_bowl","is_grasped","success","fail","dist_apple_bowl"]:
            if k in info:
                out[k] = _to_np(info.get(k))
        return out
    if isinstance(obs, dict) and "extra" in obs and isinstance(obs["extra"], dict):
        ex = obs["extra"]
        return {k:_to_np(ex.get(k)) for k in ["tcp_to_apple","apple_to_bowl","is_grasped"] if k in ex}
    return {}

def env_reset_any(env, seed=None):
    """Works with Gymnasium envs and SB3 VecEnv/VecNormalize."""
    try:
        out = env.reset(seed=seed) if seed is not None else env.reset()
    except TypeError:
        if seed is not None and hasattr(env, "env_method"):
            try:
                env.env_method("reset", seed=seed)
            except Exception:
                pass
        out = env.reset()

    if isinstance(out, tuple) and len(out) == 2:
        obs, info = out
    else:
        obs, info = out, {}
    return obs, info

def env_step_any(env, action):
    out = env.step(action)
    # Gymnasium: (obs, rew, terminated, truncated, info)
    if isinstance(out, tuple) and len(out) == 5:
        obs, rew, terminated, truncated, info = out
        done = bool(terminated or truncated)
        return obs, float(rew), done, dict(info), bool(terminated), bool(truncated)
    # VecEnv: (obs, rewards, dones, infos)
    if isinstance(out, tuple) and len(out) == 4:
        obs, rews, dones, infos = out
        rew  = _scalar(rews[0] if hasattr(rews, "__len__") else rews)
        done = bool(np.array(dones).reshape(-1)[0])
        info = (infos[0] if isinstance(infos, (list, tuple)) else infos) or {}
        truncated  = bool(info.get("TimeLimit.truncated", False)) if isinstance(info, dict) else False
        terminated = done and not truncated
        return obs, rew, done, (dict(info) if isinstance(info, dict) else {}), terminated, truncated
    raise RuntimeError(f"Unexpected env.step() format: len={len(out)}")

def evaluate_model(model, env, n_episodes=20, seed0=0, max_steps=1000, record_gif_path=None):
    rows, frames = [], []
    do_gif = (record_gif_path is not None) and (imageio is not None) and CFG["record_gif"] and RENDER_AVAILABLE

    for ep in range(n_episodes):
        obs, info = env_reset_any(env, seed=seed0 + ep * 101)
        ep_ret = 0.0
        final_info = {}
        min_t2a, min_a2b = 1e9, 1e9
        ever_grasped = False

        if do_gif and ep == 0:
            try:
                fr = env.render()
                if fr is not None:
                    frames.append(fr)
            except Exception:
                pass

        for t in range(max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rew, done, info, terminated, truncated = env_step_any(env, action)
            ep_ret += float(rew)
            final_info = dict(info)

            # ── Direct distance read from env.unwrapped (never NaN) ────────
            d_t2a = dist_tcp_apple(env)
            if d_t2a is not None:
                min_t2a = min(min_t2a, d_t2a)

            d_a2b = dist_apple_bowl(env)
            if d_a2b is not None:
                min_a2b = min(min_a2b, d_a2b)

            # is_grasped from info (still reliable)
            ex = get_extra(obs, info)
            if "is_grasped" in ex:
                ever_grasped = ever_grasped or (_scalar(ex["is_grasped"]) > 0.5)

            if do_gif and ep == 0 and len(frames) < CFG["gif_max_frames"]:
                try:
                    fr = env.render()
                    if fr is not None:
                        frames.append(fr)
                except Exception:
                    pass

            if done:
                break

        steps = t + 1

        success  = bool(np.atleast_1d(np.asarray(final_info.get("success", False))).ravel()[0])
        fail_nan = bool(np.atleast_1d(np.asarray(final_info.get("fail",    False))).ravel()[0])
        timeout  = (not success) and (not fail_nan) and (steps >= max_steps or truncated)

        rows.append({
            "episode": ep,
            "return": ep_ret,
            "steps": steps,
            "success": success,
            "fail_nan": fail_nan,
            "timeout": timeout,
            "min_dist_tcp_apple":  min_t2a if min_t2a < 1e8 else np.nan,
            "min_dist_apple_bowl": min_a2b if min_a2b < 1e8 else np.nan,
            "ever_grasped": ever_grasped,
        })

    df = pd.DataFrame(rows)

    if do_gif and len(frames) > 0:
        imageio.mimsave(record_gif_path, frames, fps=CFG["gif_fps"])
        print("🎞️ Saved GIF:", record_gif_path)

    summary = {
        "success_rate":              float(df["success"].mean()),
        "fail_nan_rate":             float(df["fail_nan"].mean()),
        "timeout_rate":              float(df["timeout"].mean()),
        "mean_return":               float(df["return"].mean()),
        "mean_steps":                float(df["steps"].mean()),
        "ever_grasped_rate":         float(df["ever_grasped"].mean()),
        "mean_min_dist_tcp_apple":   float(df["min_dist_tcp_apple"].mean(skipna=True)),
        "mean_min_dist_apple_bowl":  float(df["min_dist_apple_bowl"].mean(skipna=True)),
    }
    return df, summary


# ── Quick sanity check (only if eval_env exists) ──────────────────────────
try:
    _r = _get_raw_env(eval_env)
    _uw = _r.unwrapped
    _has_tcp   = hasattr(_uw, "_tcp_link")
    _has_apple = hasattr(_uw, "apple")
    _has_bowl  = hasattr(_uw, "bowl")
    print(f"✅ Eval utilities ready")
    print(f"   env.unwrapped has _tcp_link={_has_tcp}  apple={_has_apple}  bowl={_has_bowl}")
    if _has_tcp and _has_apple:
        eval_env.reset()
        _d = dist_tcp_apple(eval_env)
        print(f"   dist_tcp_apple (after reset): {_d}")
except NameError:
    print("✅ Eval utilities defined (eval_env not yet created)")


✅ Eval utilities ready
   env.unwrapped has _tcp_link=True  apple=True  bowl=True
   dist_tcp_apple (after reset): 0.30790629982948303


## 6) Callbacks

In [ ]:

class AppleEvalCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq, n_eval_episodes, log_dir: Path, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.log_dir = Path(log_dir)
        self.best_success = -1.0
        self.history = []
        self._next_eval = self.eval_freq

    def _on_step(self) -> bool:
        if self.num_timesteps < self._next_eval:
            return True
        self._next_eval += self.eval_freq

        gif_path = self.log_dir / f"eval_ppo_step_{self.num_timesteps}.gif"
        df, summary = evaluate_model(
            self.model, self.eval_env,
            n_episodes=self.n_eval_episodes,
            seed0=1234,
            record_gif_path=gif_path if CFG["record_gif"] else None,
        )
        df.to_csv(self.log_dir / f"eval_ppo_step_{self.num_timesteps}.csv", index=False)
        for k, v in summary.items():
            self.logger.record(f"eval/{k}", v)
        self.history.append({"timesteps": self.num_timesteps, **summary})

        if summary["success_rate"] > self.best_success:
            self.best_success = summary["success_rate"]
            self.model.save(str(self.log_dir / "best_ppo_model"))
            try:
                if isinstance(self.training_env, VecNormalize):
                    self.training_env.save(str(self.log_dir / "vecnormalize_best.pkl"))
            except Exception:
                pass
            if self.verbose:
                print(f"🏆 New best  success_rate={self.best_success:.3f} @ step {self.num_timesteps:,}")
        return True


class FailNaNMonitorCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.count = 0

    def _on_step(self) -> bool:
        for inf in self.locals.get("infos", []):
            if isinstance(inf, dict) and inf.get("fail", False):
                self.count += 1
        self.logger.record("train/fail_nan_count", float(self.count))
        return True


class RefStyleLogCallback(BaseCallback):
    """
    Prints progress in professor's reference style.
    Manually tracks episode returns (no Monitor wrapper needed).
    """
    def __init__(self, total_timesteps: int, n_steps: int, verbose=1):
        super().__init__(verbose)
        self.total_timesteps = total_timesteps
        self.n_steps   = n_steps
        self._update   = 0
        self._t0       = None
        self._ep_buf   = []
        self._cur_ret  = None

    def _on_training_start(self) -> None:
        self._t0 = time.time()
        n_envs = self.training_env.num_envs
        self._cur_ret = np.zeros(n_envs, dtype=np.float64)
        print()
        print("=" * 72)
        print(f"  PPO Training — {self.total_timesteps:,} steps  ({n_envs} envs)")
        print(f"  n_steps={self.n_steps}  batch_size={CFG['batch_size']}  "
              f"n_epochs={CFG['n_epochs']}  clip_range={CFG['clip_range']}")
        print(f"  rollout_size={n_envs * self.n_steps:,}  phaseA_steps={CFG['phaseA_steps']:,}")
        print("=" * 72)

    def _on_step(self) -> bool:
        rewards = np.asarray(self.locals.get("rewards", [0.0])).reshape(-1)
        dones   = np.asarray(self.locals.get("dones",   [False])).reshape(-1).astype(bool)
        if self._cur_ret is None:
            self._cur_ret = np.zeros(len(rewards), dtype=np.float64)
        self._cur_ret += rewards
        for i, done in enumerate(dones):
            if done:
                self._ep_buf.append(float(self._cur_ret[i]))
                self._cur_ret[i] = 0.0
        return True

    def _on_rollout_end(self) -> None:
        self._update += 1

    def _on_rollout_start(self) -> None:
        if self._update == 0 or self._t0 is None:
            return
        elapsed = time.time() - self._t0 + 1e-8
        fps     = int(self.num_timesteps / elapsed)
        mean_10 = float(np.mean(self._ep_buf[-10:])) if self._ep_buf else float("nan")
        kl_val  = self.logger.name_to_value.get("train/approx_kl", float("nan"))
        print(
            f"  Step {self.num_timesteps:>8,}/{self.total_timesteps:,} | "
            f"Update {self._update:>3} | "
            f"MeanRew(10ep): {mean_10:>8.3f} | "
            f"KL: {kl_val:.4f} | "
            f"FPS: {fps:>5}"
        )

    def _on_training_end(self) -> None:
        elapsed = time.time() - self._t0
        fps     = int(self.num_timesteps / (elapsed + 1e-8))
        mean_10 = float(np.mean(self._ep_buf[-10:])) if self._ep_buf else float("nan")
        print("=" * 72)
        print(f"  Training finished in {elapsed/60:.1f} min  ({fps} FPS)")
        print(f"  Total updates  : {self._update}")
        print(f"  Finished eps   : {len(self._ep_buf)}")
        if self._ep_buf:
            print(f"  Best ep return : {max(self._ep_buf):.4f}")
            print(f"  Last-10 mean   : {mean_10:.4f}")
        print("=" * 72)


class ProgressKPICallback(BaseCallback):
    """
    After each eval: prints KPI table.
    Warns if ever_grasped_rate=0 past phaseA_steps.
    """
    def __init__(self, eval_cb, warn_steps: int, verbose=1):
        super().__init__(verbose)
        self.eval_cb    = eval_cb
        self.warn_steps = int(warn_steps)
        self._last_ts   = -1
        self._warned    = False

    def _on_step(self) -> bool:
        if not self.eval_cb.history:
            return True
        latest = self.eval_cb.history[-1]
        ts = latest.get("timesteps", 0)
        if ts != self._last_ts:
            self._last_ts = ts
            print(f"\n  📊 KPI @ {ts:,} steps:")
            print(f"     ever_grasped_rate       : {latest.get('ever_grasped_rate', 0):.3f}")
            print(f"     mean_min_dist_tcp_apple : {latest.get('mean_min_dist_tcp_apple', float('nan')):.4f}")
            print(f"     mean_min_dist_apple_bowl: {latest.get('mean_min_dist_apple_bowl', float('nan')):.4f}")
            print(f"     success_rate            : {latest.get('success_rate', 0):.3f}")
            print(f"     mean_return             : {latest.get('mean_return', float('nan')):.4f}")

        if (not self._warned
                and self.num_timesteps >= self.warn_steps
                and not any(h.get("ever_grasped_rate", 0) > 0 for h in self.eval_cb.history)):
            self._warned = True
            print(f"\n  ⚠️  ever_grasped_rate = 0 after {self.num_timesteps:,} steps!")
            print(f"     → Consider: raise phaseB_scale_hold or check reward shaping")
        return True


class CurriculumCallback(BaseCallback):
    """
    Phase A (0 → phaseA_steps):
      ActionMask hold_scale=0.0 (hard zero), PostureHold k=phaseA_posture_k
    Phase B (phaseA_steps → ∞):
      ActionMask hold_scale=phaseB_scale_hold (soft), PostureHold k=phaseB_posture_k

    Switches at phaseA_steps OR immediately when ever_grasped_rate > 0.

    Uses ActionFilterWrapper.apply_phase_b(hold_scale, posture_k, posture_clip)
    via training_env.env_method() — works with both DummyVecEnv and SubprocVecEnv.
    """
    def __init__(self, eval_cb, cfg, verbose=1):
        super().__init__(verbose)
        self.eval_cb = eval_cb
        self.cfg     = cfg
        self._in_phase_b = False

    def _switch_to_phase_b(self):
        if self._in_phase_b:
            return
        self._in_phase_b = True

        hold_scale   = float(self.cfg["phaseB_scale_hold"])
        posture_k    = float(self.cfg["phaseB_posture_k"])
        posture_clip = float(self.cfg["phaseB_posture_clip"])

        # env_method dispatches to each subprocess's outermost gym wrapper
        # (ActionFilterWrapper). apply_phase_b() then traverses inward.
        try:
            results = self.training_env.env_method(
                "apply_phase_b", hold_scale, posture_k, posture_clip
            )
            n_ok = sum(1 for r in results if r is True)
        except Exception as e:
            n_ok = 0
            print(f"  ⚠️  apply_phase_b via env_method failed: {e}")

        if self.verbose:
            print(f"\n  🎓 Curriculum → Phase B @ {self.num_timesteps:,} steps")
            print(f"     ActionMask hold_scale : 0.0 → {hold_scale}")
            print(f"     PostureHold k         : {self.cfg['phaseA_posture_k']} → {posture_k}")
            print(f"     PostureHold clip      : {self.cfg['phaseA_posture_clip']} → {posture_clip}")
            print(f"     Updated envs: {n_ok}/{self.training_env.num_envs}")

    def _on_step(self) -> bool:
        if self.num_timesteps >= self.cfg["phaseA_steps"]:
            self._switch_to_phase_b()
            return True
        if not self._in_phase_b and self.eval_cb.history:
            if self.eval_cb.history[-1].get("ever_grasped_rate", 0) > 0:
                print(f"\n  🎓 Curriculum → Phase B EARLY (ever_grasped>0 @ {self.num_timesteps:,})")
                self._switch_to_phase_b()
        return True


# ── Instantiate all callbacks ──────────────────────────────────────────────
checkpoint_cb = CheckpointCallback(
    save_freq=max(1, CFG["checkpoint_freq"] // N_TRAIN_ENVS),
    save_path=str(ARTIFACTS_DIR),
    name_prefix="ppo_ckpt",
)
eval_cb   = AppleEvalCallback(eval_env, CFG["eval_freq"], CFG["eval_episodes"], ARTIFACTS_DIR, verbose=1)
fail_cb   = FailNaNMonitorCallback()
reflog_cb = RefStyleLogCallback(CFG["total_env_steps"], CFG["n_steps"])
kpi_cb    = ProgressKPICallback(eval_cb, warn_steps=CFG["phaseA_steps"])
curr_cb   = CurriculumCallback(eval_cb, CFG)

print("✅ Callbacks ready  (AppleEval + FailNaN + RefStyleLog + Checkpoint + KPI + Curriculum)")
print(f"   eval_freq={CFG['eval_freq']:,}  checkpoint_freq={CFG['checkpoint_freq']:,}")
print(f"   Phase B triggers at {CFG['phaseA_steps']:,} steps via env_method('apply_phase_b')")


✅ Callbacks ready  (AppleEval + FailNaN + RefStyleLog + Checkpoint + KPI + Curriculum)
   eval_freq=250,000  checkpoint_freq=500,000
   Phase B triggers at 2,000,000 steps via env_method('apply_phase_b')


## 7) Build PPO model

In [ ]:

def linear_schedule(initial_lr: float, final_lr: float):
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule

lr_schedule = linear_schedule(CFG["learning_rate"], CFG["lr_end"])

policy_kwargs = dict(
    net_arch=dict(pi=CFG["net_arch_pi"], vf=CFG["net_arch_vf"]),
    activation_fn=torch.nn.SiLU,
)

ppo_kwargs = dict(
    policy          = "MlpPolicy",
    env             = train_env,
    learning_rate   = lr_schedule,
    n_steps         = CFG["n_steps"],
    batch_size      = CFG["batch_size"],
    n_epochs        = CFG["n_epochs"],
    gamma           = CFG["gamma"],
    gae_lambda      = CFG["gae_lambda"],
    clip_range      = CFG["clip_range"],
    ent_coef        = CFG["ent_coef"],
    vf_coef         = CFG["vf_coef"],
    max_grad_norm   = CFG["max_grad_norm"],
    target_kl       = CFG["target_kl"],   # early-stop update if KL too large
    policy_kwargs   = policy_kwargs,
    verbose         = 0,                  # ref-style output via RefStyleLogCallback
    seed            = SEED,
    device          = DEVICE,
    tensorboard_log = str(ARTIFACTS_DIR / "tb"),
)

sig = inspect.signature(PPO.__init__)
if "use_sde" in sig.parameters:
    ppo_kwargs["use_sde"]          = CFG["use_sde"]
    ppo_kwargs["sde_sample_freq"]  = CFG["sde_sample_freq"]

model = PPO(**ppo_kwargs)

# Quick sanity check
obs_dim = train_env.observation_space.shape[0]
act_dim = train_env.action_space.shape[0]
param_count = sum(p.numel() for p in model.policy.parameters())
print("=" * 50)
print(f"  PPO Model Summary")
print("=" * 50)
print(f"  obs_dim        : {obs_dim}")
print(f"  act_dim        : {act_dim}  ({len(FREE)} free, {len(HOLD)} held)")
print(f"  net_arch pi    : {CFG['net_arch_pi']}")
print(f"  net_arch vf    : {CFG['net_arch_vf']}")
print(f"  Parameters     : {param_count:,}")
print(f"  Device         : {DEVICE}")
print(f"  n_steps        : {CFG['n_steps']}")
print(f"  batch_size     : {CFG['batch_size']}")
print(f"  n_epochs       : {CFG['n_epochs']}")
print(f"  clip_range     : {CFG['clip_range']}")
print(f"  target_kl      : {CFG['target_kl']}")
print(f"  use_sde        : {CFG['use_sde']}")
print("=" * 50)
print("✅ PPO model created")


  PPO Model Summary
  obs_dim        : 94
  act_dim        : 37  (12 free, 25 held)
  net_arch pi    : [512, 256, 128]
  net_arch vf    : [512, 256, 128]
  Parameters     : 430,667
  Device         : cuda
  n_steps        : 256
  batch_size     : 2048
  n_epochs       : 4
  clip_range     : 0.1
  target_kl      : 0.03
  use_sde        : False
✅ PPO model created


/root/robotic-sim-dishwash/.env/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


## 8) Train

In [ ]:

t0 = time.time()
model.learn(
    total_timesteps      = CFG["total_env_steps"],
    reset_num_timesteps  = True,   # fresh run — step counter starts at 0
    callback=[checkpoint_cb, eval_cb, fail_cb, reflog_cb, kpi_cb, curr_cb],
    progress_bar=False,
)



  PPO Training — 12,000,000 steps  (32 envs)
  n_steps=256  batch_size=2048  n_epochs=4  clip_range=0.1
  rollout_size=8,192  phaseA_steps=2,000,000


  Step    8,192/12,000,000 | Update   1 | MeanRew(10ep):      nan | KL: 0.0003 | FPS:   226
  Step   16,384/12,000,000 | Update   2 | MeanRew(10ep):      nan | KL: 0.0002 | FPS:   287
  Step   24,576/12,000,000 | Update   3 | MeanRew(10ep):      nan | KL: 0.0001 | FPS:   315
  Step   32,768/12,000,000 | Update   4 | MeanRew(10ep):    0.064 | KL: 0.0002 | FPS:   335
  Step   40,960/12,000,000 | Update   5 | MeanRew(10ep):    0.064 | KL: 0.0002 | FPS:   345
  Step   49,152/12,000,000 | Update   6 | MeanRew(10ep):    0.064 | KL: 0.0001 | FPS:   355
  Step   57,344/12,000,000 | Update   7 | MeanRew(10ep):    0.064 | KL: 0.0002 | FPS:   363
  Step   65,536/12,000,000 | Update   8 | MeanRew(10ep):    0.252 | KL: 0.0003 | FPS:   368
  Step   73,728/12,000,000 | Update   9 | MeanRew(10ep):    0.252 | KL: 0.0003 | FPS:   372
  Step   81,920/12,000,000 | Update  10 | MeanRew(10ep):    0.252 | KL: 0.0002 | FPS:   371
  Step   90,112/12,000,000 | Update  11 | MeanRew(10ep):    0.252 | KL: 0.0003 |

KeyboardInterrupt: 

## 9) Save final artifacts

In [ ]:

model.save(str(ARTIFACTS_DIR / "ppo_final_model"))
print("✅ Saved:", ARTIFACTS_DIR / "ppo_final_model.zip")

try:
    if isinstance(train_env, VecNormalize):
        train_env.save(str(ARTIFACTS_DIR / "vecnormalize_final.pkl"))
        print("✅ Saved VecNormalize stats")
except Exception:
    pass

hist = pd.DataFrame(eval_cb.history)
hist.to_csv(ARTIFACTS_DIR / "eval_history.csv", index=False)
print("✅ Saved eval_history.csv")


## 10) Plot curves

In [ ]:

hist_path = ARTIFACTS_DIR / "eval_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    if len(hist) > 0:
        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(hist["timesteps"], hist["success_rate"], marker="o")
        ax.set_title("PPO success_rate over training")
        ax.set_xlabel("timesteps")
        ax.set_ylabel("success_rate")
        fig.tight_layout()
        fig.savefig(ARTIFACTS_DIR / "plot_success_rate.png", dpi=160)
        plt.show()

        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(hist["timesteps"], hist["mean_return"], marker="o")
        ax.set_title("PPO mean_return over training")
        ax.set_xlabel("timesteps")
        ax.set_ylabel("mean_return (normalized_dense)")
        fig.tight_layout()
        fig.savefig(ARTIFACTS_DIR / "plot_mean_return.png", dpi=160)
        plt.show()


## 11) Final eval + error analysis

In [ ]:

final_gif = ARTIFACTS_DIR / "ppo_final_eval.gif"
df_eval, summary = evaluate_model(model, eval_env, n_episodes=CFG["eval_episodes"], seed0=555, record_gif_path=final_gif)
df_eval.to_csv(ARTIFACTS_DIR / "ppo_final_eval_episodes.csv", index=False)
(ARTIFACTS_DIR / "ppo_final_eval_summary.json").write_text(json.dumps(summary, indent=2))

print("Final eval summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

def bucket(row):
    if row["fail_nan"]:
        return "fail_nan"
    if row["success"]:
        return "success"
    if row["timeout"]:
        return "timeout"
    dA = row["min_dist_tcp_apple"]
    grasp = row["ever_grasped"]
    dB = row["min_dist_apple_bowl"]
    if np.isnan(dA) or dA > 0.12:
        return "no_reach"
    if not grasp:
        return "reach_no_grasp"
    if np.isnan(dB) or dB > 0.15:
        return "grasp_no_place"
    return "other_fail"

df_eval["fail_type"] = df_eval.apply(bucket, axis=1)
fail_summary = df_eval["fail_type"].value_counts().rename_axis("fail_type").reset_index(name="count")
fail_summary.to_csv(ARTIFACTS_DIR / "ppo_fail_summary.csv", index=False)
print("\nFailure breakdown:")
print(fail_summary)
